In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import math
import platform

df = pd.read_csv("data/2025_Airbnb_NYC_listings.csv") #----- 자기 경로 설정!!
df_cleaned = pd.read_csv("first_clean_data.csv")

In [27]:
# 데이터 처리 및 분석
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
from scipy.stats import skew, kurtosis
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg
import scikit_posthocs as sp

# 머신러닝
from sklearn.preprocessing import MinMaxScaler

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [28]:
df_machine = df_cleaned.copy()
df_machine.head()

,Unnamed: 0,id,name,description,host_id,host_since,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bedrooms,beds,amenities,price,availability_365,number_of_reviews,number_of_reviews_ltm,estimated_occupancy_l365d,estimated_revenue_l365d,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,log_price
0,0,36121,Lg Rm in Historic Prospect Heights,Cozy space share in the heart of a great neigh...,62165,2009-12-11,-1.0,-1.0,87.0,False,Prospect Heights,Brooklyn,40.673760,-73.966110,Private room in rental unit,Private room,1,1.0,1.0,"[""Refrigerator"", ""Dishes and silverware"", ""Wif...",200,362,9,0,0,0.0,4.88,5.00,4.80,5.00,5.00,5.00,5.00,1,0,1,0,0.05,5.303305
1,1,36647,"1 Bedroom & your own Bathroom, Elevator Apartment",Private bedroom with your own bathroom in a 2 ...,157798,2010-07-04,-1.0,-1.0,100.0,False,East Harlem,Manhattan,40.792454,-73.940742,Private room in condo,Private room,2,1.0,1.0,"[""Oven"", ""Blender"", ""Luggage dropoff allowed"",...",82,204,102,0,0,0.0,4.77,4.82,4.76,4.88,4.90,4.38,4.71,1,0,1,0,0.58,4.418841
2,2,38663,Luxury Brownstone in Boerum Hill,"Beautiful, large home in great hipster neighbo...",165789,2010-07-13,3.0,100.0,40.0,False,Boerum Hill,Brooklyn,40.684420,-73.980680,Private room in home,Private room,2,5.0,5.0,"[""Portable fans"", ""Oven"", ""Baking sheet"", ""Fir...",765,326,43,0,0,0.0,4.70,4.83,4.52,4.88,4.88,4.86,4.62,1,0,1,0,0.28,6.641182
3,3,38833,Spectacular West Harlem Garden Apt,This is a very large and unique space. An inc...,166532,2010-07-14,4.0,100.0,97.0,True,Harlem,Manhattan,40.818058,-73.946671,Entire home,Entire home/apt,2,1.0,1.0,"[""Fire extinguisher"", ""Clothing storage: close...",139,25,241,42,255,35445.0,4.85,4.87,4.50,4.96,4.96,4.79,4.82,1,1,0,0,1.36,4.941642
4,4,39282,“Work-from-home” from OUR home.,*Monthly Discount will automatically apply <br...,168525,2010-07-16,4.0,100.0,100.0,True,Williamsburg,Brooklyn,40.710651,-73.950874,Private room in rental unit,Private room,2,1.0,1.0,"[""Oven"", ""Rice maker"", ""Laundromat nearby"", ""L...",130,38,274,12,154,20020.0,4.82,4.83,4.61,4.94,4.88,4.85,4.78,2,0,2,0,1.54,4.875197


In [29]:
drop_cols = ['name', 'description', 'neighbourhood_cleansed', 'number_of_reviews', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication',
            'review_scores_location', 'review_scores_value','estimated_occupancy_l365d','estimated_revenue_l365d','calculated_host_listings_count_shared_rooms','reviews_per_month',
            'calculated_host_listings_count','calculated_host_listings_count_entire_homes','calculated_host_listings_count_private_rooms','host_id','host_since','id','latitude',
            'longitude','amenities','property_type']
df_machine=df_machine.drop(columns = drop_cols)
df_machine.shape


(22248, 16)

In [30]:
host_is_idx = df_machine.loc[df_machine['host_is_superhost'] == 'unknown'].index
df_machine = df_machine.drop(host_is_idx)

In [31]:
df_machine['host_is_superhost'].value_counts()

host_is_superhost
False    15746
True      6129
Name: count, dtype: int64

In [32]:
neighborhood_idx = df_machine.loc[df_machine['neighbourhood_group_cleansed'].isin(['Bronx','Staten Island'])].index
df_machine = df_machine.drop(neighborhood_idx)

In [33]:
df_machine['neighbourhood_group_cleansed'].value_counts()

neighbourhood_group_cleansed
Manhattan    10008
Brooklyn      7328
Queens        3325
Name: count, dtype: int64

In [34]:
room_idx = df_machine.loc[df_machine['room_type'].isin(['Hotel room','Shared room'])].index
df_machine = df_machine.drop(room_idx)

In [35]:
df_machine['room_type'].value_counts()

room_type
Entire home/apt    11868
Private room        8358
Name: count, dtype: int64

In [36]:
df_machine.shape

(20226, 16)

In [37]:
from sklearn.model_selection import train_test_split
# 타겟(y)에 NaN이 있는 행은 학습에서 제외 version 1
df_final = df_machine.dropna(subset=['review_scores_rating'])

X = df_final.drop(columns=["review_scores_rating"])
y = df_final["review_scores_rating"]
# X = df_machine.drop(columns=["review_scores_rating"])
# y = df_machine["review_scores_rating"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [38]:
col = "review_scores_accuracy"
X_train[f"{col}_isna"] = X_train[col].isna().astype(int)
X_test[f"{col}_isna"]  = X_test[col].isna().astype(int)

In [39]:
# 2. 원 핫 인코딩
cat_cols = ["neighbourhood_group_cleansed", "room_type","host_is_superhost"]
X_train_dum = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_test_dum = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)
# 3. reindex(=컬럼 정렬)
X_test_dum = X_test_dum.reindex(columns=X_train_dum.columns, fill_value=0)

In [40]:
X_train_dum.columns

Index(['Unnamed: 0', 'host_response_time', 'host_response_rate',
       'host_acceptance_rate', 'accommodates', 'bedrooms', 'beds', 'price',
       'availability_365', 'number_of_reviews_ltm', 'review_scores_accuracy',
       'log_price', 'review_scores_accuracy_isna',
       'neighbourhood_group_cleansed_Manhattan',
       'neighbourhood_group_cleansed_Queens', 'room_type_Private room',
       'host_is_superhost_True'],
      dtype='str')

In [41]:
X_test_dum.columns

Index(['Unnamed: 0', 'host_response_time', 'host_response_rate',
       'host_acceptance_rate', 'accommodates', 'bedrooms', 'beds', 'price',
       'availability_365', 'number_of_reviews_ltm', 'review_scores_accuracy',
       'log_price', 'review_scores_accuracy_isna',
       'neighbourhood_group_cleansed_Manhattan',
       'neighbourhood_group_cleansed_Queens', 'room_type_Private room',
       'host_is_superhost_True'],
      dtype='str')

In [42]:
X_train_dum.info()

<class 'pandas.DataFrame'>
Index: 11258 entries, 7801 to 8643
Data columns (total 17 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Unnamed: 0                              11258 non-null  int64  
 1   host_response_time                      11258 non-null  float64
 2   host_response_rate                      11258 non-null  float64
 3   host_acceptance_rate                    11258 non-null  float64
 4   accommodates                            11258 non-null  int64  
 5   bedrooms                                11258 non-null  float64
 6   beds                                    11258 non-null  float64
 7   price                                   11258 non-null  int64  
 8   availability_365                        11258 non-null  int64  
 9   number_of_reviews_ltm                   11258 non-null  int64  
 10  review_scores_accuracy                  11258 non-null  float64
 11  log

In [43]:
# price는 극단값이 꽤 있어서 로그 변환을 같이 쓰는 경우가 많음
X_train_dum["price_log1p"] = np.log1p(X_train_dum["price"])  # log(1+price)
X_test_dum["price_log1p"] = np.log1p(X_test_dum["price"])

In [44]:
from sklearn.preprocessing import StandardScaler

scale_cols = ["price_log1p", "host_response_time", "host_response_rate", "host_acceptance_rate","accommodates","bedrooms","beds","availability_365","number_of_reviews_ltm","review_scores_accuracy",
            "review_scores_accuracy_isna"]
scale_cols = [c for c in scale_cols if c in X_train_dum.columns]  # 안전장치

scaler = StandardScaler()
X_train_dum[scale_cols] = scaler.fit_transform(X_train_dum[scale_cols])
X_test_dum[scale_cols] = scaler.transform(X_test_dum[scale_cols])

X_train_dum[scale_cols].describe()

,price_log1p,host_response_time,host_response_rate,host_acceptance_rate,accommodates,bedrooms,beds,availability_365,number_of_reviews_ltm,review_scores_accuracy,review_scores_accuracy_isna
count,1.125800e+04,1.125800e+04,1.125800e+04,1.125800e+04,1.125800e+04,1.125800e+04,1.125800e+04,1.125800e+04,1.125800e+04,1.125800e+04,11258.0
mean,-4.613668e-16,9.593400e-17,-8.930698e-17,-1.186552e-16,1.735648e-17,-3.029495e-17,1.893434e-17,6.311447e-17,1.262289e-17,6.071612e-16,0.0
std,1.000044e+00,1.000044e+00,1.000044e+00,1.000044e+00,1.000044e+00,1.000044e+00,1.000044e+00,1.000044e+00,1.000044e+00,1.000044e+00,0.0
min,-2.879224e+00,-1.911017e+00,-1.909117e+00,-3.099972e+00,-9.948585e-01,-1.435296e+00,-1.455020e+00,-2.014561e+00,-2.956567e-01,-8.621430e+00,0.0
25%,-6.610313e-01,-3.230948e-01,6.887982e-02,-2.113371e-01,-4.807143e-01,-3.100480e-01,-6.025734e-01,-8.919869e-01,-2.956567e-01,-1.268560e-01,0.0
50%,-3.600564e-02,7.355202e-01,5.881040e-01,2.508444e-01,-4.807143e-01,-3.100480e-01,-6.025734e-01,1.504035e-01,-2.621181e-01,2.863936e-01,0.0
75%,6.025310e-01,7.355202e-01,5.881040e-01,7.515411e-01,5.475741e-01,8.152004e-01,2.498733e-01,9.789703e-01,-1.279638e-01,5.618933e-01,0.0
max,5.861921e+00,7.355202e-01,5.881040e-01,7.515411e-01,6.717304e+00,1.544343e+01,1.644636e+01,1.237341e+00,5.953717e+01,5.618933e-01,0.0


In [45]:
# 중앙값으로 결측치를 채움 (isna 컬럼이 있으므로 정보 손실 걱정 없음)
accuracy_median = X_train_dum['review_scores_accuracy'].median()

X_train_dum['review_scores_accuracy'] = X_train_dum['review_scores_accuracy'].fillna(accuracy_median)
X_test_dum['review_scores_accuracy'] = X_test_dum['review_scores_accuracy'].fillna(accuracy_median)

In [46]:
# 학습에 사용하지 않을 중복/불필요 컬럼 제거
cols_to_drop = ['log_price', 'Unnamed: 0'] 
X_train_final = X_train_dum.drop(columns=[c for c in cols_to_drop if c in X_train_dum.columns])
X_test_final = X_test_dum.drop(columns=[c for c in cols_to_drop if c in X_test_dum.columns])

In [1]:
import xgboost as xgb
from sklearn.metrics import r2_score, mean_absolute_error

# 1) XGBoost 회귀 모델 생성
# n_estimators: 나무의 개수
# learning_rate: 학습 속도 (너무 높으면 과적합, 너무 낮으면 학습이 느림)
# max_depth: 나무의 깊이 (정교함을 결정)
model_xgb = xgb.XGBRegressor(
    n_estimators=1000, 
    learning_rate=0.05, 
    max_depth=6, 
    random_state=42, 
    n_jobs=-1,
    objective='reg:squarederror' # 회귀 분석 설정
)

# 2) 모델 학습
# XGBoost는 학습 과정에서 검증 데이터를 보고 스스로 조기 종료(Early Stopping)할 수도 있습니다.
model_xgb.fit(
    X_train_dum, y_train,
    eval_set=[(X_test_dum, y_test)],
    verbose=False
)

# 3) 예측
y_train_pred = model_xgb.predict(X_train_dum)
y_test_pred = model_xgb.predict(X_test_dum)

# 4) 결과 출력 (질문자님이 원하시는 형식)
print("✅ XGBoost 모델 학습 및 평가 완료")
print("-" * 30)
print(f"Train R2: {r2_score(y_train, y_train_pred):.4f} | Test R2: {r2_score(y_test, y_test_pred):.4f}")
print(f"Train MAE: {mean_absolute_error(y_train, y_train_pred):.4f} | Test MAE: {mean_absolute_error(y_test, y_test_pred):.4f}")

NameError: name 'X_train_dum' is not defined